In [325]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [326]:
PROJECT_ROOT = "/Users/ashritkuma.samudrala/lnex/ex_llm_rag"

In [327]:
import sys
sys.path.insert(0, f"{PROJECT_ROOT}/main/src")
from utils import read_file

In [ ]:
path = f"{PROJECT_ROOT}/main/resources/books/tinyshakespeare.txt"
# path = f"{PROJECT_ROOT}/main/resources/mb.txt"
text = read_file(path)
print(len(text))
text[:100]

1115394


'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [358]:
text_set = set(text)
vocab = list(sorted(text_set))
vocab_size = len(vocab)
print(len(vocab))
print("".join(vocab))
print(vocab)

65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [357]:
# Hyper parms
context_len  = 8 # aka block_size, sequence_len, etc..
batch_size = 32
max_train_iters = 5000
loss_eval_interval = 100
loss_eval_itrs = 100
emb_dim = 32
attn_head_size = 32
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

Setup encoder decoders

In [330]:
c2i = {ch: i for i, ch in enumerate(vocab)}
i2c = {i: ch for i, ch in enumerate(vocab)}

def encode(test: str):
    return [c2i[ch] for ch in test]

def decode(tokens: list[int]):
    return ''.join([i2c[i] for i in tokens])

In [240]:
encode("how are you")
decode([46, 53, 61, 1, 39, 56, 43, 1, 63, 53, 59])

'how are you'

In [331]:
text_encoded = encode(text)
len(text_encoded)
tokens = torch.tensor(text_encoded, dtype=torch.long)
tokens[:100]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

In [242]:
total_toks = len(tokens)
train = int(0.9 * total_toks)
train_data = tokens[:train]
val_data = tokens[train:]

Setup batching and data sets

In [332]:
def get_batch(split=None):
    data = val_data if split == 'val' else train_data
    random_batch_starts = torch.randint(len(data) - context_len, (batch_size,))
    x = torch.stack([data[i:i+context_len] for i in random_batch_starts])
    y = torch.stack([data[i+1:i+context_len+1] for i in random_batch_starts])
    x,y = x.to(device), y.to(device)
    return x, y

    

In [245]:
x,y = get_batch()
x.shape
y.shape

torch.Size([32, 8])

Lets  start with a simple model

In [128]:
# this is functionally equvivalent to a bigram model.
# The embdding lookup is of sixe (V,V) 
#  each token index maps to a row of size V, which serves directly as the logits over the next token. This is exactly what a bigram model does: given the current token, predict the
# next token using only that single token's statistics.
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, vocab_size) # (V, V)
    
    def forward(self, x, targets=None): # x => (B, CL), targets => (B, CL)
        logits = self.embedding(x) # (B, CL, V)
        if targets is not None:
            B, CL, V = logits.shape
            # cross entropy expects either (B*CL, V) or (B, V, CL) so we need to convert
            logits = logits.view(B*CL, V)
            targets = targets.view(B*CL)
            loss = F.cross_entropy(logits, targets)
        else:
            loss = None
        return logits, loss
    
    def generate(self, inp, max_len): # inp (B, CL)
        for _ in range(max_len):
            logits, loss = self.forward(inp) # (B, CL, V)
            # We only need the last token logits in every batch
            logits = logits[:, -1, :] # (B, V)

            # Apply SFMX along the last dim, i.e last token raw logits in every batch
            probs = F.softmax(logits, dim=-1) # (B, V)

            # sample, this gives next token index for every batch
            next_token = torch.multinomial(probs, num_samples=1) # (B, 1)
            inp = torch.cat([inp, next_token], dim=1) # (B, CL+1)
        
        return inp

model = SimpleModel()
model.to(device)

            



SimpleModel(
  (embedding): Embedding(91, 91)
)

In [ ]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval() # put in eval mode
    for split in ['train', 'val']:
        losses = torch.zeros(loss_eval_itrs)
        for k in range(loss_eval_itrs):
            x,y = get_batch(split)
            logits, loss = model(x,y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # put back in train mode, note this does do training just puts the model in training mode
    return out

In [130]:
# Optimzation
def train_model():
    optim = torch.optim.Adam(model.parameters(), lr=1e-3)
    for i in range(max_train_iters):
        x,y = get_batch()
        logits, loss = model(x,y)
        optim.zero_grad()
        loss.backward()
        optim.step()

        if (i % loss_eval_interval == 0):
            out = estimate_loss()
            print(f"Itration {i}: Train Loss: {out['train']:.4f}, Val Loss: {out['val']:.4f}")
        



In [134]:
train_model()

Itration 0: Train Loss: 2.4632, Val Loss: 2.4906
Itration 100: Train Loss: 2.4658, Val Loss: 2.4841
Itration 200: Train Loss: 2.4717, Val Loss: 2.4953
Itration 300: Train Loss: 2.4595, Val Loss: 2.4719
Itration 400: Train Loss: 2.4630, Val Loss: 2.4882
Itration 500: Train Loss: 2.4639, Val Loss: 2.4863
Itration 600: Train Loss: 2.4612, Val Loss: 2.4792
Itration 700: Train Loss: 2.4573, Val Loss: 2.4912
Itration 800: Train Loss: 2.4598, Val Loss: 2.4933
Itration 900: Train Loss: 2.4644, Val Loss: 2.4696
Itration 1000: Train Loss: 2.4585, Val Loss: 2.4862
Itration 1100: Train Loss: 2.4763, Val Loss: 2.4904
Itration 1200: Train Loss: 2.4639, Val Loss: 2.4977
Itration 1300: Train Loss: 2.4495, Val Loss: 2.4765
Itration 1400: Train Loss: 2.4621, Val Loss: 2.4789
Itration 1500: Train Loss: 2.4654, Val Loss: 2.4890
Itration 1600: Train Loss: 2.4610, Val Loss: 2.4752
Itration 1700: Train Loss: 2.4581, Val Loss: 2.4847
Itration 1800: Train Loss: 2.4613, Val Loss: 2.4966
Itration 1900: Train Los

In [169]:
i = encode("\n")
inp = torch.tensor([[i[0]]], dtype=torch.long, device=device)
print(inp)
# inp = torch.zeros((1,1), dtype=torch.long, device=device)

print(inp.shape)
op_tox = model.generate(inp, 100)
print(decode(op_tox[0].tolist()))

tensor([[0]], device='mps:0')
torch.Size([1, 1])

C6;D9X[ QN]QN X]Q LTN QR], a\Y[N\N JWLNJK^U O _NN-
6B*
.W KN\ 0<CXW \]QX ]QJ[]c VNU LQN R\ LQ&
.A1D8


In [ ]:
#### Scratch
test = nn.Embedding(vocab_size, vocab_size)
l = test(x)

# print(l.shape)
# d = l[:, -1, :]
# print(d.shape)
# g = l[:, :, -1]
# print(g.shape)

t1 = torch.tensor([[1,2,3]])
t2 = torch.tensor([[4,5,6]])
xxx = torch.cat((t1, t2), dim=1)
print(xxx)
print(xxx.shape)

print("="*10)

xxx = torch.cat([t1, t2], dim=0)
print(xxx)
print(xxx.shape)

tensor([[1, 2, 3, 4, 5, 6]])
torch.Size([1, 6])
tensor([[1, 2, 3],
        [4, 5, 6]])
torch.Size([2, 3])


Look at @attention_intuition.ipynb for more details on attn

In [255]:
# Hyper parms
context_len  = 8 # aka block_size, sequence_len, etc..
batch_size = 32
max_train_iters = 5000
loss_eval_interval = 100
loss_eval_itrs = 100
emb_dim = 32
attn_head_size = 32
n_heads = 4
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
# Lets create a simple GPT model


class AttnHead(nn.Module):
    def __init__(self, attn_head_size):
        super().__init__()
        self.Qw = nn.Linear(emb_dim, attn_head_size)
        self.Kw = nn.Linear(emb_dim, attn_head_size)
        self.Vw = nn.Linear(emb_dim, attn_head_size)
        self.register_buffer('tril', torch.tril(torch.ones(context_len, context_len)))
    
    def forward(self, x): # X (B, CL, ED)
        B, CL, ED = x.shape
        Q = self.Qw(x) # (B, CL, attn_head_size)
        K = self.Kw(x) # (B, CL, attn_head_size)
        V = self.Vw(x) # (B, CL, attn_head_size)
        
        w = Q @ K.transpose(1, 2) # (B, CL, attn_head_size) @ (B, attn_head_size, CL) = (B, CL, CL)
        w = w * (attn_head_size ** -0.5)
        # We need to do :CL, :CL instead of sequence_len, since in generation we might have less than context_len size
        w = w.masked_fill(self.tril[:CL, :CL] == 0, float('-inf'))
        w = F.softmax(w, dim=-1)
        
        # Perform weighted aggregation of values
        out = w @ V # (B, CL, CL) @ (B, CL, attn_head_size) = (B, CL, attn_head_size)
        return out


# It helps to have multiple channels of communicatin (heads) per attention block so that each of the head can commnicate separately and gather different kinds of data and finally they are all mixed together
class MultiHeadAttn(nn.Module):
    def __init__(self, n_heads):
        super().__init__()
        self.heads = nn.ModuleList([AttnHead(emb_dim//n_heads) for _ in range(n_heads)]) # creates 4 4 self attn heads each 8 dimensional
    
    def forward(self, x): # (B, CL, ED)
        return torch.cat([head(x) for head in self.heads], dim=-1) # head(x) would return (B, CL, attn_head_size), concatinating all of them over last dim attn_head_size would give us back (B, CL, ED)
        
        

class GPTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim) # (V, ED)
        self.positions = nn.Embedding(context_len, emb_dim) # (CL, ED)
        self.lm_head = nn.Linear(emb_dim, vocab_size) # (ED, V)
        self.m_head_attn = MultiHeadAttn(n_heads)
    
    def forward(self, x, targets=None): # x => (B, CL), targets => (B, CL)
        B, CL = x.shape
        tok_emb = self.embedding(x) # (B, CL, ED)
        pos_emb = self.positions(torch.arange(CL, device=device)) # (CL, ED), torch.arange create 0 to CL-1 ints, we pluck those indexes out of positions
        # print(f"Tok emb = {tok_emb.shape}, pos emb = {pos_emb.shape}")
        x = tok_emb + pos_emb # (B, CL, ED)
        x = self.m_head_attn(x) # (B, CL, ED)
        logits = self.lm_head(x) # (B, CL, V)
        if targets is not None:
            B, CL, V = logits.shape
            # cross entropy expects either (B*CL, V) or (B, V, CL) so we need to convert
            logits = logits.view(B*CL, V)
            targets = targets.view(B*CL)
            loss = F.cross_entropy(logits, targets)
        else:
            loss = None
        return logits, loss
    
    def generate(self, inp, max_len): # inp (B, CL)
        for _ in range(max_len):
            # Crop till context length
            inp_ctx = inp[:, -context_len:]
            logits, _ = self.forward(inp_ctx) # (B, CL, V)
            # We only need the last token logits in every batch
            logits = logits[:, -1, :] # (B, V)

            # Apply SFMX along the last dim, i.e last token raw logits in every batch
            probs = F.softmax(logits, dim=-1) # (B, V)

            # sample, this gives next token index for every batch
            next_token = torch.multinomial(probs, num_samples=1) # (B, 1)
            inp = torch.cat([inp, next_token], dim=1) # (B, CL+1)
        
        return inp

gpt_model = GPTModel()
gpt_model.to(device)

GPTModel(
  (embedding): Embedding(65, 32)
  (positions): Embedding(8, 32)
  (lm_head): Linear(in_features=32, out_features=65, bias=True)
  (m_head_attn): MultiHeadAttn(
    (heads): ModuleList(
      (0-3): 4 x AttnHead(
        (Qw): Linear(in_features=32, out_features=8, bias=True)
        (Kw): Linear(in_features=32, out_features=8, bias=True)
        (Vw): Linear(in_features=32, out_features=8, bias=True)
      )
    )
  )
)

In [334]:
# Optimzation
def train_gpt_model(model, learning_rate=1e-3):
    optim = torch.optim.Adam(model.parameters(), lr=learning_rate)
    for i in range(max_train_iters):
        x,y = get_batch()
        logits, loss = model(x,y)
        optim.zero_grad()
        loss.backward()
        optim.step()

        if (i % loss_eval_interval == 0):
            out = estimate_loss(model)
            print(f"Itration {i}: Train Loss: {out['train']:.4f}, Val Loss: {out['val']:.4f}")
        

In [ ]:
train_gpt_model(gpt_model)

Itration 0: Train Loss: 2.2538, Val Loss: 2.2569
Itration 100: Train Loss: 2.2359, Val Loss: 2.2674
Itration 200: Train Loss: 2.2544, Val Loss: 2.2663
Itration 300: Train Loss: 2.2232, Val Loss: 2.2653
Itration 400: Train Loss: 2.2358, Val Loss: 2.2749
Itration 500: Train Loss: 2.2366, Val Loss: 2.2500
Itration 600: Train Loss: 2.2291, Val Loss: 2.2600
Itration 700: Train Loss: 2.2316, Val Loss: 2.2534
Itration 800: Train Loss: 2.2138, Val Loss: 2.2758
Itration 900: Train Loss: 2.2146, Val Loss: 2.2458
Itration 1000: Train Loss: 2.2207, Val Loss: 2.2510
Itration 1100: Train Loss: 2.2258, Val Loss: 2.2534
Itration 1200: Train Loss: 2.2314, Val Loss: 2.2504
Itration 1300: Train Loss: 2.2181, Val Loss: 2.2698
Itration 1400: Train Loss: 2.2162, Val Loss: 2.2491
Itration 1500: Train Loss: 2.2178, Val Loss: 2.2570
Itration 1600: Train Loss: 2.2105, Val Loss: 2.2416
Itration 1700: Train Loss: 2.2201, Val Loss: 2.2474
Itration 1800: Train Loss: 2.1959, Val Loss: 2.2324
Itration 1900: Train Los

In [263]:
i = encode(" ")
inp = torch.tensor([[i[0]]], dtype=torch.long, device=device)
# print(inp)
# inp = torch.zeros((1,1), dtype=torch.long, device=device)

# print(inp.shape)
op_tox = gpt_model.generate(inp, 200)
# print(op_tox.shape)
print(decode(op_tox[0].tolist()))

 hith city Pecom for forwing: sesomarens sinch mould, muselve sey hes. CBENTER:
He:
Spearknicthy bike.
 coansalcorth stos,
Thout molave matior dilllesse,
Fors, our anocrced, wn sto tair, arw'lled him p


* Now we initially had a singe attention head of emb_dim size
* We created multi head attn with n_heads of emb_dim size
* We then concatenated the outputs of all the heads and passed it through a linear layer to get the final output of emb_dim size
* This helps in parallel processing of the data and also helps in capturing different types of relationships between the tokens
* But after that we are directly getting the logits, there is no processing of the information gathered from the attention.
* Here is where we need an MLP layer to process the information and then get the logits.

In [265]:
# Lets add MLP to the GPT

class FeedForwardNetwork(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.ffn = nn.Sequential(
            # nn.Linear(emb_dim, 4 * emb_dim),
            # nn.ReLU(),
            # nn.Linear(4 * emb_dim, emb_dim)
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.ffn(x)


class GPTModel1(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim) # (V, ED)
        self.positions = nn.Embedding(context_len, emb_dim) # (CL, ED)
        self.lm_head = nn.Linear(emb_dim, vocab_size) # (ED, V)
        self.m_head_attn = MultiHeadAttn(n_heads)
        self.ffn = FeedForwardNetwork(emb_dim)
    
    def forward(self, x, targets=None): # x => (B, CL), targets => (B, CL)
        B, CL = x.shape
        tok_emb = self.embedding(x) # (B, CL, ED)
        pos_emb = self.positions(torch.arange(CL, device=device)) # (CL, ED), torch.arange create 0 to CL-1 ints, we pluck those indexes out of positions
        # print(f"Tok emb = {tok_emb.shape}, pos emb = {pos_emb.shape}")
        x = tok_emb + pos_emb # (B, CL, ED)
        x = self.m_head_attn(x) # (B, CL, ED)
        x = self.ffn(x) # (B, CL, ED)
        logits = self.lm_head(x) # (B, CL, V)
        if targets is not None:
            B, CL, V = logits.shape
            # cross entropy expects either (B*CL, V) or (B, V, CL) so we need to convert
            logits = logits.view(B*CL, V)
            targets = targets.view(B*CL)
            loss = F.cross_entropy(logits, targets)
        else:
            loss = None
        return logits, loss
    
    def generate(self, inp, max_len): # inp (B, CL)
        for _ in range(max_len):
            # Crop till context length
            inp_ctx = inp[:, -context_len:]
            logits, _ = self.forward(inp_ctx) # (B, CL, V)
            # We only need the last token logits in every batch
            logits = logits[:, -1, :] # (B, V)

            # Apply SFMX along the last dim, i.e last token raw logits in every batch
            probs = F.softmax(logits, dim=-1) # (B, V)

            # sample, this gives next token index for every batch
            next_token = torch.multinomial(probs, num_samples=1) # (B, 1)
            inp = torch.cat([inp, next_token], dim=1) # (B, CL+1)
        
        return inp

gpt_model1 = GPTModel1()
gpt_model1.to(device)

GPTModel1(
  (embedding): Embedding(65, 32)
  (positions): Embedding(8, 32)
  (lm_head): Linear(in_features=32, out_features=65, bias=True)
  (m_head_attn): MultiHeadAttn(
    (heads): ModuleList(
      (0-3): 4 x AttnHead(
        (Qw): Linear(in_features=32, out_features=8, bias=True)
        (Kw): Linear(in_features=32, out_features=8, bias=True)
        (Vw): Linear(in_features=32, out_features=8, bias=True)
      )
    )
  )
  (ffn): FeedForwardNetwork(
    (ffn): Sequential(
      (0): Linear(in_features=32, out_features=32, bias=True)
      (1): ReLU()
    )
  )
)

In [271]:
train_gpt_model(gpt_model1)

Itration 0: Train Loss: 2.2296, Val Loss: 2.2613
Itration 100: Train Loss: 2.2193, Val Loss: 2.2530
Itration 200: Train Loss: 2.2164, Val Loss: 2.2452
Itration 300: Train Loss: 2.2006, Val Loss: 2.2444
Itration 400: Train Loss: 2.2147, Val Loss: 2.2265
Itration 500: Train Loss: 2.2010, Val Loss: 2.2275
Itration 600: Train Loss: 2.2057, Val Loss: 2.2368
Itration 700: Train Loss: 2.2080, Val Loss: 2.2204
Itration 800: Train Loss: 2.1981, Val Loss: 2.2296
Itration 900: Train Loss: 2.1905, Val Loss: 2.2472
Itration 1000: Train Loss: 2.2030, Val Loss: 2.2273
Itration 1100: Train Loss: 2.1893, Val Loss: 2.2255
Itration 1200: Train Loss: 2.1972, Val Loss: 2.2364
Itration 1300: Train Loss: 2.1772, Val Loss: 2.2348
Itration 1400: Train Loss: 2.1754, Val Loss: 2.2303
Itration 1500: Train Loss: 2.1901, Val Loss: 2.1957
Itration 1600: Train Loss: 2.1826, Val Loss: 2.2039
Itration 1700: Train Loss: 2.1937, Val Loss: 2.2112
Itration 1800: Train Loss: 2.1869, Val Loss: 2.2243
Itration 1900: Train Los

In [273]:
i = encode("a")
inp = torch.tensor([[i[0]]], dtype=torch.long, device=device)
# print(inp.shape)
op_tox = gpt_model.generate(inp, 200)
# print(op_tox.shape)
print(decode(op_tox[0].tolist()))

auk ap of am urave. Dordsarthe, herd ark gour ford, k ond:
amey huir abtoon wiy der imn:
Fwely, incstim to wolt are's ongo; pice hinterlo.

Murce wire gor. sesd so thimry'd efand hat.

YOR werd, baysea


* We have achieved good loss with multi head attn followed by MLP
* The combination one MHA and MLP is called a transformer block
* We would need multiple trsnformer blocks, so that each block can learn different patterns

In [ ]:
# Lets add multiple transformer blocks

class TransformerBlock(nn.Module):
    def __init__(self, n_heads, emb_dim):
        super().__init__()
        self.m_head_attn = MultiHeadAttn(n_heads)
        self.ffn = FeedForwardNetwork(emb_dim)
    
    def forward(self, x):
        x = self.m_head_attn(x)
        x = self.ffn(x)
        return x

class GPTModel2(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim) # (V, ED)
        self.positions = nn.Embedding(context_len, emb_dim) # (CL, ED)
        
        self.blocks = nn.Sequential(
            TransformerBlock(n_heads, emb_dim),
            TransformerBlock(n_heads, emb_dim),
            TransformerBlock(n_heads, emb_dim)
        )

        self.lm_head = nn.Linear(emb_dim, vocab_size) # (ED, V)
    
    def forward(self, x, targets=None): # x => (B, CL), targets => (B, CL)
        B, CL = x.shape
        tok_emb = self.embedding(x) # (B, CL, ED)
        pos_emb = self.positions(torch.arange(CL, device=device)) # (CL, ED), torch.arange create 0 to CL-1 ints, we pluck those indexes out of positions
        # print(f"Tok emb = {tok_emb.shape}, pos emb = {pos_emb.shape}")
        x = tok_emb + pos_emb # (B, CL, ED)
        x = self.blocks(x) # (B, CL, ED)
        logits = self.lm_head(x) # (B, CL, V)
        if targets is not None:
            B, CL, V = logits.shape
            # cross entropy expects either (B*CL, V) or (B, V, CL) so we need to convert
            logits = logits.view(B*CL, V)
            targets = targets.view(B*CL)
            loss = F.cross_entropy(logits, targets)
        else:
            loss = None
        return logits, loss
    
    def generate(self, inp, max_len): # inp (B, CL)
        for _ in range(max_len):
            # Crop till context length
            inp_ctx = inp[:, -context_len:]
            logits, _ = self.forward(inp_ctx) # (B, CL, V)
            # We only need the last token logits in every batch
            logits = logits[:, -1, :] # (B, V)

            # Apply SFMX along the last dim, i.e last token raw logits in every batch
            probs = F.softmax(logits, dim=-1) # (B, V)

            # sample, this gives next token index for every batch
            next_token = torch.multinomial(probs, num_samples=1) # (B, 1)
            inp = torch.cat([inp, next_token], dim=1) # (B, CL+1)
        
        return inp

gpt_model2 = GPTModel2()
gpt_model2.to(device)

GPTModel2(
  (embedding): Embedding(65, 32)
  (positions): Embedding(8, 32)
  (blocks): Sequential(
    (0): TransformerBlock(
      (m_head_attn): MultiHeadAttn(
        (heads): ModuleList(
          (0-3): 4 x AttnHead(
            (Qw): Linear(in_features=32, out_features=8, bias=True)
            (Kw): Linear(in_features=32, out_features=8, bias=True)
            (Vw): Linear(in_features=32, out_features=8, bias=True)
          )
        )
      )
      (ffn): FeedForwardNetwork(
        (ffn): Sequential(
          (0): Linear(in_features=32, out_features=32, bias=True)
          (1): ReLU()
        )
      )
    )
    (1): TransformerBlock(
      (m_head_attn): MultiHeadAttn(
        (heads): ModuleList(
          (0-3): 4 x AttnHead(
            (Qw): Linear(in_features=32, out_features=8, bias=True)
            (Kw): Linear(in_features=32, out_features=8, bias=True)
            (Vw): Linear(in_features=32, out_features=8, bias=True)
          )
        )
      )
      (ffn): F

In [275]:
train_gpt_model(gpt_model2)

Itration 0: Train Loss: 4.1949, Val Loss: 4.1966
Itration 100: Train Loss: 3.2809, Val Loss: 3.3512
Itration 200: Train Loss: 3.2817, Val Loss: 3.3051
Itration 300: Train Loss: 3.2316, Val Loss: 3.2823
Itration 400: Train Loss: 3.1639, Val Loss: 3.1446
Itration 500: Train Loss: 3.0838, Val Loss: 3.0661
Itration 600: Train Loss: 2.9579, Val Loss: 2.9663
Itration 700: Train Loss: 2.8646, Val Loss: 2.8514
Itration 800: Train Loss: 2.7953, Val Loss: 2.7953
Itration 900: Train Loss: 2.7530, Val Loss: 2.7317
Itration 1000: Train Loss: 2.7252, Val Loss: 2.6927
Itration 1100: Train Loss: 2.6716, Val Loss: 2.6687
Itration 1200: Train Loss: 2.6544, Val Loss: 2.6490
Itration 1300: Train Loss: 2.6027, Val Loss: 2.6147
Itration 1400: Train Loss: 2.5894, Val Loss: 2.5669
Itration 1500: Train Loss: 2.5633, Val Loss: 2.5525
Itration 1600: Train Loss: 2.5475, Val Loss: 2.5450
Itration 1700: Train Loss: 2.5391, Val Loss: 2.5357
Itration 1800: Train Loss: 2.5277, Val Loss: 2.5007
Itration 1900: Train Los

In [277]:
i = encode("a")
inp = torch.tensor([[i[0]]], dtype=torch.long, device=device)
# print(inp.shape)
op_tox = gpt_model.generate(inp, 200)
# print(op_tox.shape)
print(decode(op_tox[0].tolist()))

aw-
Bo, willdettill, youll Hesded thoing vif not corw deimssinin Athy agerthe is, hy, no-an hasen.
Noy barre
And, fomey abtears, wocrthe kpepris of! and I kiencthoul his my no?
Andays. That?

Hownoulse


### BatchNorm vs LayerNorm

---
  What they normalize over:

  * BatchNorm1d (your torchfied_char_mlp_with_bn.py):
    * cur_mean = input.mean(0, keepdim=True)   # mean across the BATCH dimension
    * Input (32, 100) → mean shape (1, 100)
    * "For each neuron, what's the average activation across all 32 examples?"

  * LayerNorm (your gpt_dev.ipynb):
    * cur_mean = input.mean(1, keepdim=True)   # mean across the FEATURE dimension
    * Input (32, 100) → mean shape (32, 1)
    * "For each example, what's the average activation across all 100 features?"

  BatchNorm normalizes across the batch (per neuron).
  LayerNorm normalizes across the features (per sample).

  ---
  Why BatchNorm doesn't work well for transformers:

  1. Variable sequence lengths. Your batches have shape (B, CL, ED). At generation time, you pass sequences of length 1, 2, 3... up to context_len. BatchNorm would compute statistics over the batch AND sequence
  dimensions — those statistics change with sequence length, making behavior inconsistent between training and inference.

  2. Small/single-sample inference. BatchNorm's running stats (your running_mean/running_var) are an approximation. With batch size=1 at generation time, the batch statistics are meaningless — you'd just be normalizing
   a single number. LayerNorm doesn't have this problem since it normalizes within each sample independently.

  3. No running stats needed. Look at your LayerNorm.__call__ — it doesn't have a running_mean/running_var buffer. It always computes stats from the current input directly, so it behaves identically in train and eval.
  Your BatchNorm1d needs those buffers precisely because at eval time the batch might be too small to get reliable statistics.

  ---
  Summary:
  
  | Feature | BatchNorm | LayerNorm |
| :--- | :--- | :--- |
| **Normalizes over** | Batch dim (across examples) | Feature dim (within each example) |
| **Needs running stats** | Yes | No |
| **Works at batch size=1** | Poor (Fails during training) | Fine |
| **Works with variable seq len** | Poor | Fine |
| **Good for** | CNNs, MLPs | Transformers, RNNs |
  

In [ ]:
class LayerNorm:
    """
        dim: Number of dimensions or no of outputs (neurons) af the layer after which this batch norm is applied
        momentum: Used to calculate the exp moving avg of running_mean and running_variance
    """
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.momentum = momentum
        self.eps = eps
        self.dim = dim
        self.training = True

        # define the learnable parms gain(gamma) and bias(beta).
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    # applies the layer norm
    # idea here is to normalize across the last dimension (rows), scale it by gain and shift it by bias
    def __call__(self, input: torch.Tensor): # (B, CL, ED)
        # print(f"LayerNorm: Input shape = {input.shape}, gain shape = {self.gamma.shape}, bias shape = {self.beta.shape}")
        # we need to use the mean and variance of the input in training mode, when in eval or sampling we need to use the running_mean and running_vae
        if self.training:
            # we are calculating this over 1 dimension(along the rows, i.e across all inputs for every neuron). So a (32,100) input will become (32, 1)
            cur_mean = input.mean(1, keepdim=True)
            cur_var = input.var(1, keepdim=True)
        else:
            cur_mean = self.running_mean
            cur_var = self.running_var
        
        # standardize input to unit variance (z_score standardization)
        iHat = (input - cur_mean)/torch.sqrt(cur_var + self.eps)
        # scale by the gain and add bias. At the initialization since gain is vector of 1's and bias is vector of 0's it out will be perfect gaussian dist with unit variance.
        # since gain(gamma) and bias(beta) are learnable parms they will be modified in the back prop and will be adjusted accordingly to minimize the loss
        self.out = self.gamma * iHat + self.beta
        
        return self.out
    
    def parameters(self):
        return [self.gamma, self.beta]

# xd = torch.randn(32,100)
# ln = LayerNorm(100)
# odd = ln(xd)
# print(odd.shape)
# print(f"Column 0: mean {odd[:, 0].mean():.6f}, std {odd[:, 0].std():.6f}") # Columns
# print(f"Row 0: mean {odd[0, :].mean():.6f}, std {odd[0, :].std():.6f}") # Rows
# Layer norm mormalized across rows


* Another importent addition to this is having residual connection
* This means, we add the input to the output of the transformer block, this is useful because addition distributes the gradinets equally to the nodes when doing back prop
* In the initilization the transformer block won't contribute much to the output, but as we train the model, the transformer block will learn to contribute more to the output. 
* So during the initial phases, the gradient directly flows to the input not impeded by the transformer block
* This helps normalize the gradients

### Claude explanation
 ---                                                                                                                                                                                                                     
  1. Residual connections (x = x + self.m_head_attn(x))                                                                                                                                                                 
                                                                                                                                                                                                                          
  The key intuition: at initialization, the transformer block produces near-random outputs. Without residuals, the gradient has to flow through the attention/FFN layers to reach the embeddings — through a noisy, random
   path. With residuals:

  x = x + block(x)

  The +x creates a highway for gradients to flow directly back to the embedding layer, bypassing the block entirely. Early in training, block(x) is noise, so the model essentially ignores it and relies on x alone (like
   a bigram). As training progresses, the block gradually learns to add useful corrections on top of x. The model learns incrementally rather than all-at-once.

  ---
  2. self.proj = nn.Linear(emb_dim, emb_dim) (the W0 projection)

  Each head operates in its own emb_dim//n_heads = 8 dimensional subspace. After concatenating all 4 heads you get back (B, CL, 32), but these are just 4 independent subspaces stacked — they haven't been mixed.

  The projection layer lets the model blend information across heads. Without it, head 0's output can never influence how head 1's output is used. The projection is essentially: "now that all heads have gathered their
  information, combine it into a coherent representation." It also makes the residual connection mathematically clean — the output is properly scaled for x + out.

  ---
  3. Expand → contract in FFN (emb_dim → 4*emb_dim → emb_dim)

  The attention layer is good at routing information (which tokens talk to which). But it's essentially a weighted average — limited in what computations it can express.

  The FFN is where the model does per-token computation. The expansion to 4*emb_dim creates a wider intermediate space where the ReLU can activate different combinations of features. Think of it as: attention decides
  what to look at, FFN decides what to do with it.

  The 4x factor comes from the original "Attention is All You Need" paper and is empirically found to work well — wide enough to have expressive power, contracted back so the residual addition stays in the same emb_dim
   space.

  Without the expansion (your earlier FeedForwardNetwork using emb_dim → emb_dim), the FFN has very limited capacity — it's nearly just a linear transformation with a ReLU, which is why GPTModel2 trains slower and
  worse than GPTModel3.

In [335]:
# Hyper parms


context_len  = 256 # aka block_size, sequence_len, etc..
batch_size = 64
learning_rate = 3e-4
max_train_iters = 5000
loss_eval_interval = 500
loss_eval_itrs = 200
emb_dim = 384
n_heads = 6
n_transformer_blocks = 6
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
dropout = 0.2

In [ ]:
# Lets add resudial connections and layer norm

class AttnHead_N(nn.Module):
    def __init__(self, attn_head_size):
        super().__init__()
        self.Qw = nn.Linear(emb_dim, attn_head_size)
        self.Kw = nn.Linear(emb_dim, attn_head_size)
        self.Vw = nn.Linear(emb_dim, attn_head_size)
        self.register_buffer('tril', torch.tril(torch.ones(context_len, context_len)))

        # This will randomly swuth off some neurons (dropout%) during training to prevent overfitting
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x): # X (B, CL, ED)
        B, CL, ED = x.shape
        Q = self.Qw(x) # (B, CL, attn_head_size)
        K = self.Kw(x) # (B, CL, attn_head_size)
        V = self.Vw(x) # (B, CL, attn_head_size)
        
        w = Q @ K.transpose(1, 2) # (B, CL, attn_head_size) @ (B, attn_head_size, CL) = (B, CL, CL)
        w = w * (attn_head_size ** -0.5)
        # We need to do :CL, :CL instead of sequence_len, since in generation we might have less than context_len size
        w = w.masked_fill(self.tril[:CL, :CL] == 0, float('-inf'))
        w = F.softmax(w, dim=-1)

        # we drop out before finally getting the attention o/p so that we can switch off few neurons from comunicating (prevent overfitting)
        w = self.dropout(w)
        
        # Perform weighted aggregation of values
        out = w @ V # (B, CL, CL) @ (B, CL, attn_head_size) = (B, CL, attn_head_size)
        return out


# It helps to have multiple channels of communicatin (heads) per attention block so that each of the head can commnicate separately and gather different kinds of data and finally they are all mixed together
class MultiHeadAttn_N(nn.Module):
    def __init__(self, n_heads):
        super().__init__()
        self.heads = nn.ModuleList([AttnHead_N(emb_dim//n_heads) for _ in range(n_heads)]) # creates 4 4 self attn heads each 8 dimensional

        # This also called W0.
        # This will linearly mixes (transform) the attention output
        self.proj = nn.Linear(emb_dim, emb_dim)

        # This will randomly swuth off some neurons (dropout%) during training to prevent overfitting
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x): # (B, CL, ED)
        out = torch.cat([head(x) for head in self.heads], dim=-1) # head(x) would return (B, CL, attn_head_size), concatinating all of them over last dim attn_head_size would give us back (B, CL, ED)
        out = self.proj(out) #(B, CL, ED)
        out = self.dropout(out)
        return out


class FeedForwardNetwork_N(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.ffn(x)

class TransformerBlock_N(nn.Module):
    def __init__(self, n_heads, emb_dim):
        super().__init__()
        self.m_head_attn = MultiHeadAttn_N(n_heads)
        self.ffn = FeedForwardNetwork_N(emb_dim)
        # layer norm before attention
        self.ln1 = nn.LayerNorm(emb_dim)

        # layer norm before mlp
        self.ln2 = nn.LayerNorm(emb_dim)
    
    def forward(self, x):
        x = x + self.m_head_attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class GPTModel3(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim) # (V, ED)
        self.positions = nn.Embedding(context_len, emb_dim) # (CL, ED)
        
        self.blocks = nn.Sequential(
            *[TransformerBlock_N(n_heads, emb_dim) for _ in range(n_transformer_blocks)] # '*' like the spread (...) operator in JS, spreads the blocks in the list
        )

        # Final layer normalization
        self.ln_f = nn.LayerNorm(emb_dim)

        self.lm_head = nn.Linear(emb_dim, vocab_size) # (ED, V)
    
    def forward(self, x, targets=None): # x => (B, CL), targets => (B, CL)
        B, CL = x.shape
        tok_emb = self.embedding(x) # (B, CL, ED)
        pos_emb = self.positions(torch.arange(CL, device=device)) # (CL, ED), torch.arange create 0 to CL-1 ints, we pluck those indexes out of positions
        # print(f"Tok emb = {tok_emb.shape}, pos emb = {pos_emb.shape}")
        x = tok_emb + pos_emb # (B, CL, ED)
        x = self.blocks(x) # (B, CL, ED)
        x = self.ln_f(x) # (B, CL, ED)
        logits = self.lm_head(x) # (B, CL, V)
        if targets is not None:
            B, CL, V = logits.shape
            # cross entropy expects either (B*CL, V) or (B, V, CL) so we need to convert
            logits = logits.view(B*CL, V)
            targets = targets.view(B*CL)
            loss = F.cross_entropy(logits, targets)
        else:
            loss = None
        return logits, loss
    
    def generate(self, inp, max_len, temperature=1.0): # inp (B, CL)
        for _ in range(max_len):
            # Crop till context length
            inp_ctx = inp[:, -context_len:]
            logits, _ = self.forward(inp_ctx) # (B, CL, V)
            # We only need the last token logits in every batch
            logits = logits[:, -1, :] # (B, V)

            # Apply SFMX along the last dim, i.e last token raw logits in every batch
            probs = F.softmax(logits, dim=-1) # (B, V)

            # sample, this gives next token index for every batch
            next_token = torch.multinomial(probs, num_samples=1) # (B, 1)
            inp = torch.cat([inp, next_token], dim=1) # (B, CL+1)
        
        return inp

gpt_model3 = GPTModel3()
gpt_model3.to(device)

GPTModel3(
  (embedding): Embedding(65, 384)
  (positions): Embedding(256, 384)
  (blocks): Sequential(
    (0): TransformerBlock_N(
      (m_head_attn): MultiHeadAttn_N(
        (heads): ModuleList(
          (0-5): 6 x AttnHead_N(
            (Qw): Linear(in_features=384, out_features=64, bias=True)
            (Kw): Linear(in_features=384, out_features=64, bias=True)
            (Vw): Linear(in_features=384, out_features=64, bias=True)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffn): FeedForwardNetwork_N(
        (ffn): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
    

In [338]:
train_gpt_model(gpt_model3, learning_rate)

Itration 0: Train Loss: 3.6095, Val Loss: 3.6412
Itration 500: Train Loss: 1.8446, Val Loss: 1.9584
Itration 1000: Train Loss: 1.5119, Val Loss: 1.6894
Itration 1500: Train Loss: 1.3825, Val Loss: 1.5924
Itration 2000: Train Loss: 1.3040, Val Loss: 1.5375
Itration 2500: Train Loss: 1.2488, Val Loss: 1.5145
Itration 3000: Train Loss: 1.2035, Val Loss: 1.5040
Itration 3500: Train Loss: 1.1635, Val Loss: 1.4902
Itration 4000: Train Loss: 1.1287, Val Loss: 1.4948
Itration 4500: Train Loss: 1.0896, Val Loss: 1.4894


In [354]:
i = encode("thou")
inp = torch.tensor([i], dtype=torch.long, device=device)
print(inp.shape)
op_tox = gpt_model3.generate(inp, 200)
# print(op_tox.shape)
print(decode(op_tox[0].tolist()))

torch.Size([1, 4])
thou wast but, to denied tribunes.
Take none confess this is a widowl-mine
Which crides.
Who, my lord, 'tis worship to this remorse,
Whose age, in resign arms:
I am sortening glass show the time sings
Tha


In [355]:
import os
save_path = f"{PROJECT_ROOT}/main/resources/models/char_model/gpt_char_model3_5000iters.model"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    'model_state_dict': gpt_model3.state_dict(),
    'hyperparams': {
        'context_len': context_len,
        'emb_dim': emb_dim,
        'n_heads': n_heads,
        'n_transformer_blocks': n_transformer_blocks,
        'dropout': dropout,
        'vocab_size': vocab_size,
    },
    'train_loss': 1.0896,   # from your last log
    'val_loss': 1.4894,
    'iters': 5000,
}, save_path)

print(f"Saved to {save_path}")


Saved to /Users/ashritkuma.samudrala/lnex/ex_llm_rag/main/resources/models/char_model/gpt_char_model3_5000iters.model


In [ ]:
# Load:
checkpoint = torch.load(save_path, map_location=device)

# Restore hyperparams first
hp = checkpoint['hyperparams']
context_len = hp['context_len']
emb_dim = hp['emb_dim']
n_heads = hp['n_heads']
n_transformer_blocks = hp['n_transformer_blocks']
dropout = hp['dropout']

# Rebuild model and load weights
loaded_model = GPTModel3()
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.to(device)
loaded_model.eval()

print(f"Loaded model — train loss: {checkpoint['train_loss']}, val loss: {checkpoint['val_loss']}")

In [7]:
print(float(3e-3))

0.003
